# DML Lab Assignment No. 7
## Generate Image Examples using GANs (Generative Adversarial Networks)
**Dataset:** MNIST handwritten digits

### Step 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import mnist

print('TensorFlow version:', tf.__version__)

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Hyperparameters
LATENT_DIM  = 100    # Noise vector dimension for Generator
IMG_SHAPE   = (28, 28, 1)
BATCH_SIZE  = 128
EPOCHS      = 50

# Load and preprocess MNIST
(X_train, _), (_, _) = mnist.load_data()
X_train = X_train.astype('float32')
# Scale to [-1, 1] for tanh output in Generator
X_train = (X_train - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=-1)   # (60000, 28, 28, 1)

print('Dataset shape:', X_train.shape)

### Step 2A: Design the Generator
Takes a random noise vector and generates a fake image.

In [ ]:
def build_generator(latent_dim):
    model = models.Sequential(name='Generator')
    
    # Foundation: dense layer -> reshape to small feature map
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Reshape((7, 7, 256)))
    
    # Upsample to 14x14
    model.add(layers.Conv2DTranspose(128, (5,5), strides=(1,1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))
    
    # Upsample to 28x28
    model.add(layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))
    
    # Output layer: 28x28x1 image, tanh activation
    model.add(layers.Conv2DTranspose(1, (5,5), strides=(2,2), padding='same',
                                     use_bias=False, activation='tanh'))
    
    assert model.output_shape == (None, 28, 28, 1)
    return model

generator = build_generator(LATENT_DIM)
generator.summary()

### Step 2B: Design the Discriminator
Classifies images as Real (1) or Fake (0).

In [ ]:
def build_discriminator(img_shape):
    model = models.Sequential(name='Discriminator')
    
    model.add(layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=img_shape))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(128, (5,5), strides=(2,2), padding='same'))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))   # Real=1 / Fake=0
    
    return model

discriminator = build_discriminator(IMG_SHAPE)
discriminator.summary()

### Step 3: Define Optimizers and Loss Functions
- **Loss:** Binary Cross-Entropy
- **Optimizer (Generator):** Adam with lr=2e-4, beta_1=0.5
- **Optimizer (Discriminator):** Adam with lr=2e-4, beta_1=0.5

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy()

gen_optimizer  = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
disc_optimizer = optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

def discriminator_loss(real_output, fake_output):
    """Discriminator should output 1 for real, 0 for fake."""
    real_loss = cross_entropy(tf.ones_like(real_output),  real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    """Generator wants Discriminator to classify fakes as real."""
    return cross_entropy(tf.ones_like(fake_output), fake_output)

print('Optimizers and loss functions defined.')

### Training Step (using tf.GradientTape for custom training loop)

In [ ]:
@tf.function
def train_step(real_images):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)
        
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)
        
        gen_loss  = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    
    gen_grads  = gen_tape.gradient(gen_loss,  generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    gen_optimizer.apply_gradients(zip(gen_grads,  generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))
    
    return gen_loss, disc_loss

### Helper: Visualize Generated Images

In [ ]:
FIXED_NOISE = tf.random.normal([16, LATENT_DIM])   # Fixed noise for consistent visualization

def generate_and_plot(epoch, fixed_noise, save=False):
    generated = generator(fixed_noise, training=False)
    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for i, ax in enumerate(axes.flatten()):
        # Rescale from [-1,1] to [0,1]
        img = (generated[i, :, :, 0].numpy() + 1) / 2
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    plt.suptitle(f'Generated Images — Epoch {epoch}')
    plt.tight_layout()
    plt.show()

### Step 3: Train the GAN

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(X_train)\
            .shuffle(60000).batch(BATCH_SIZE, drop_remainder=True)

gen_losses  = []
disc_losses = []

print(f'Starting GAN training for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    epoch_gen_loss  = []
    epoch_disc_loss = []
    
    for real_batch in dataset:
        g_loss, d_loss = train_step(real_batch)
        epoch_gen_loss.append(g_loss.numpy())
        epoch_disc_loss.append(d_loss.numpy())
    
    avg_g = np.mean(epoch_gen_loss)
    avg_d = np.mean(epoch_disc_loss)
    gen_losses.append(avg_g)
    disc_losses.append(avg_d)
    
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Gen Loss: {avg_g:.4f} | Disc Loss: {avg_d:.4f}')
        generate_and_plot(epoch, FIXED_NOISE)

print('Training complete!')

### Step 4: Plot Loss Value vs Epoch

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS+1), gen_losses,  label='Generator Loss',     color='blue')
plt.plot(range(1, EPOCHS+1), disc_losses, label='Discriminator Loss', color='red')
plt.title('GAN Training Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Cross-Entropy)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Step 5: Test Model Performance — Final Generated Images

In [ ]:
# Generate brand-new images using random noise
test_noise = tf.random.normal([25, LATENT_DIM])
final_imgs  = generator(test_noise, training=False)

plt.figure(figsize=(8, 8))
for i in range(25):
    plt.subplot(5, 5, i+1)
    img = (final_imgs[i, :, :, 0].numpy() + 1) / 2
    plt.imshow(img, cmap='gray')
    plt.axis('off')
plt.suptitle('Final GAN Generated MNIST Digits', fontsize=14)
plt.tight_layout()
plt.show()

# Compare real vs generated side by side
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    # Real
    real = (X_train[i, :, :, 0] + 1) / 2
    axes[0, i].imshow(real, cmap='gray')
    axes[0, i].set_title('Real', fontsize=8)
    axes[0, i].axis('off')
    # Fake
    fake = (final_imgs[i, :, :, 0].numpy() + 1) / 2
    axes[1, i].imshow(fake, cmap='gray')
    axes[1, i].set_title('Generated', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Real vs GAN-Generated Images')
plt.tight_layout()
plt.show()